# Notebook 11 — Held-Out Test Set Evaluation

Loads the trained LightGBM model and evaluates it on the 2023–2024 held-out test set using a strict chronological split. No data from 2023+ was seen during training or hyperparameter tuning.

**Split:**
- Train: 2017–2021
- Val: 2022 (used for early stopping in notebook 09)
- **Test: 2023–2024 (evaluated here)**

**Reads:** `final_dataset.parquet`, `lgbm_model.pkl`, `baseline_model.pkl`, `full_model_metrics.json`  
**Writes:** `data/processed/test_metrics.json`

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
FIGURES_DIR = Path("../figures")

weekly = pd.read_parquet(PROCESSED_DIR / "final_dataset.parquet")
weekly["week_start"] = pd.to_datetime(weekly["week_start"])

with open(MODELS_DIR / "lgbm_model.pkl", "rb") as f:
    model = pickle.load(f)

with open(MODELS_DIR / "baseline_model.pkl", "rb") as f:
    baseline = pickle.load(f)

with open(PROCESSED_DIR / "full_model_metrics.json", "r") as f:
    train_metrics = json.load(f)

with open(PROCESSED_DIR / "baseline_metrics.json", "r") as f:
    baseline_metrics_file = json.load(f)

FEATURE_COLS = train_metrics["features"]
BASELINE_FEATURES = baseline_metrics_file["features"]

print(f"Dataset: {len(weekly)} rows")
print(f"Model features: {len(FEATURE_COLS)}")
print(f"Baseline features: {len(BASELINE_FEATURES)}")

## Chronological Split

In [ ]:
train = weekly[weekly["week_start"] < "2022-01-01"]
val   = weekly[(weekly["week_start"] >= "2022-01-01") & (weekly["week_start"] < "2023-01-01")]
test  = weekly[weekly["week_start"] >= "2023-01-01"]

print(f"Train: {len(train):>4} rows  {train['week_start'].min().date()} → {train['week_start'].max().date()}")
print(f"Val:   {len(val):>4} rows  {val['week_start'].min().date()} → {val['week_start'].max().date()}")
print(f"Test:  {len(test):>4} rows  {test['week_start'].min().date()} → {test['week_start'].max().date()}")

## Score on Test Set

In [ ]:
def poisson_deviance(y_true, y_pred):
    y_pred = np.maximum(y_pred, 1e-10)
    return 2 * np.mean(y_true * np.log(np.maximum(y_true, 1e-10) / y_pred) - (y_true - y_pred))

def score(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    dev  = poisson_deviance(y_true.values, y_pred)
    # Correlation between predicted and actual
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    print(f"  {label}")
    print(f"    MAE:             {mae:.4f}")
    print(f"    RMSE:            {rmse:.4f}")
    print(f"    Poisson Dev:     {dev:.4f}")
    print(f"    Pearson r:       {corr:.4f}")
    return dict(mae=mae, rmse=rmse, poisson_deviance=dev, pearson_r=corr)

lgbm_test_preds     = model.predict(test[FEATURE_COLS])
baseline_test_preds = baseline.predict(test[BASELINE_FEATURES])
lgbm_val_preds      = model.predict(val[FEATURE_COLS])

print("=== Validation set (2022) ===")
val_scores = score(val["stranding_count"], lgbm_val_preds, "LightGBM")

print("\n=== Held-out test set (2023–2024) ===")
test_lgbm     = score(test["stranding_count"], lgbm_test_preds,     "LightGBM")
print()
test_baseline = score(test["stranding_count"], baseline_test_preds, "Baseline (Poisson GLM)")

mae_improvement = (test_baseline["mae"] - test_lgbm["mae"]) / test_baseline["mae"] * 100
print(f"\nLightGBM vs Baseline MAE improvement on test: {mae_improvement:.1f}%")

## Per-Region Test Scores

In [ ]:
test_results = test[["week_start", "region", "stranding_count"]].copy()
test_results["lgbm_pred"]     = lgbm_test_preds
test_results["baseline_pred"] = baseline_test_preds

region_scores = []
for region, grp in test_results.groupby("region"):
    lgbm_mae = mean_absolute_error(grp["stranding_count"], grp["lgbm_pred"])
    base_mae = mean_absolute_error(grp["stranding_count"], grp["baseline_pred"])
    region_scores.append({
        "region": region,
        "n_weeks": len(grp),
        "mean_actual": grp["stranding_count"].mean(),
        "lgbm_mae": lgbm_mae,
        "baseline_mae": base_mae,
        "improvement_pct": (base_mae - lgbm_mae) / base_mae * 100,
    })

region_df = pd.DataFrame(region_scores).set_index("region")
region_df = region_df.sort_values("lgbm_mae", ascending=False)
print(region_df.round(3).to_string())

## Predictions vs Actuals Over Time

In [ ]:
regions = sorted(test_results["region"].unique())
fig, axes = plt.subplots(len(regions), 1, figsize=(14, 3 * len(regions)), sharex=True)

for ax, region in zip(axes, regions):
    grp = test_results[test_results["region"] == region].sort_values("week_start")
    ax.plot(grp["week_start"], grp["stranding_count"], label="Actual", color="black", lw=1.5)
    ax.plot(grp["week_start"], grp["lgbm_pred"],     label="LightGBM", color="steelblue", lw=1.2, alpha=0.85)
    ax.plot(grp["week_start"], grp["baseline_pred"], label="Baseline", color="tomato",    lw=1.0, ls="--", alpha=0.7)
    ax.set_title(region)
    ax.set_ylabel("Strandings")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Week")
plt.suptitle("Test Set (2023–2024): Predicted vs Actual Strandings", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_predictions_vs_actuals.png", dpi=150, bbox_inches="tight")
plt.show()

## Residual Distribution

In [ ]:
test_results["residual"] = test_results["stranding_count"] - test_results["lgbm_pred"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(test_results["residual"], bins=30, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="black", ls="--")
axes[0].set_title("Residuals (actual − predicted)")
axes[0].set_xlabel("Residual")

axes[1].scatter(test_results["lgbm_pred"], test_results["residual"],
                alpha=0.4, s=15, color="steelblue")
axes[1].axhline(0, color="black", ls="--")
axes[1].set_title("Residuals vs Predicted")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean residual (bias): {test_results['residual'].mean():.4f}")
print(f"Std residual:         {test_results['residual'].std():.4f}")

## Save Test Metrics

In [ ]:
test_metrics = {
    "split": {"train": "<2022", "val": "2022", "test": "2023–2024"},
    "test_rows": len(test),
    "lgbm": test_lgbm,
    "baseline": test_baseline,
    "lgbm_vs_baseline_mae_improvement_pct": float(mae_improvement),
    "per_region": region_df.round(4).to_dict(orient="index"),
}

with open(PROCESSED_DIR / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved test_metrics.json")
print(f"\nFinal test MAE (LightGBM): {test_lgbm['mae']:.4f}")
print(f"Val MAE (from training):   {train_metrics['val_mae']:.4f}")
print(f"Gap (test − val):          {test_lgbm['mae'] - train_metrics['val_mae']:+.4f}")